# Exploring a press corpus and comparing tokenizers

The corpus of these labs: every 1914 issue of two French daily newspapers, *L'Humanité* (360 issues) and *Le Figaro* (24 issues), digitized by the French national library (BnF). The text comes from raw optical character recognition (OCR): it has the flaws of a real company corpus. The texts are in French; the code works the same on any language.

1. Load the corpus and measure its statistics
2. Find and fix the OCR's flaws, then split into passages
3. Compare three tokenizers, and estimate what processing the corpus costs

## Data

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s french-press-1914
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
%pip install -q wordfreq

## 1. Loading the corpus

Each issue is a JSON file, `french-press-1914/<newspaper>/1914/<YYYYMMDD>/<YYYYMMDD>.metadata.fulltext.json`, with three fields of interest:

- `title`: the newspaper's name, in a list (`content["title"][0]`)
- `date`: the publication date, in a list (`content["date"][0]`)
- `contentAsText`: the text, one string per page, lines separated by `\n`

Build a `pandas.DataFrame` named `issues`, one row per issue, with the columns `newspaper`, `date` (as a date) and `text` (the pages joined with `\n`). Show the number of issues per newspaper and the distribution of the number of words per issue.

In [ ]:
# Your code here

### Solution

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("french-press-1914").rglob("*.json")):
    content = json.loads(path.read_text(encoding="utf8"))
    rows.append((content["title"][0], content["date"][0], "\n".join(content["contentAsText"])))
issues = pd.DataFrame(rows, columns=["newspaper", "date", "text"])
issues["date"] = pd.to_datetime(issues["date"])

print(issues.newspaper.value_counts())
issues.text.str.split().str.len().describe()

Count the corpus's word forms, lowercased, with the regular expression `WORD` below and a `collections.Counter`. Show:

- the total number of words and the number of distinct forms
- the share of forms that appear only once (*hapax legomena*)
- the 20 most frequent forms

In [ ]:
import re

WORD = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)?")

In [ ]:
# Your code here

### Solution

In [ ]:
from collections import Counter

counts = Counter(word.lower() for text in issues.text for word in WORD.findall(text))
total = sum(counts.values())
hapax = sum(1 for count in counts.values() if count == 1)
print(f"{total:,} words, {len(counts):,} forms, {hapax / len(counts):.0%} hapax legomena")
counts.most_common(20)

Plot each form's frequency against its rank, on log scales, with Zipf's law's line (the first word's frequency divided by the rank). Three quarters of the forms are hapax legomena: what do you think they are?

In [ ]:
# Your code here

### Solution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

frequencies = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(frequencies) + 1)
plt.loglog(ranks, frequencies, label="French press of 1914")
plt.loglog(ranks, frequencies[0] / ranks, "--", label="Zipf's law")
plt.xlabel("Rank")
plt.ylabel("Occurrences")
plt.legend()
plt.show()

# Many hapax legomena are OCR errors: misread, split or merged words.

## 2. The OCR's flaws

Words split at a line end stayed split: « con- tinuer ». Count the matches of the regular expression `SPLIT` below and show a dozen of them.

In [ ]:
SPLIT = re.compile(r"([^\W\d_]+)-\s+([^\W\d_]+)")

In [ ]:
# Your code here

### Solution

In [ ]:
matches = [match for text in issues.text for match in SPLIT.finditer(text)]
print(len(matches))
[match[0] for match in matches[:10]]

Rejoining every pair would be wrong: « Sud- Ouest » must stay two words. The [`wordfreq`](https://github.com/rspeer/wordfreq) library gives a word's frequency in French: `zipf_frequency(word, "fr")` is 0 for an unknown word.

Write a function `rejoin(text)` that only rejoins a pair if the joined word is known, then count the pairs rejoined and those left as they are. The `known` function remembers its answers: the corpus has millions of words.

In [ ]:
from functools import cache

from wordfreq import zipf_frequency


@cache
def known(word: str) -> bool:
    return zipf_frequency(word.lower(), "fr") > 0

In [ ]:
# Your code here

### Solution

In [ ]:
def rejoin(text: str) -> str:
    def fix(match: re.Match[str]) -> str:
        joined = match[1] + match[2]
        return joined if known(joined) else match[0]

    return SPLIT.sub(fix, text)


rejoined = sum(known(match[1] + match[2]) for match in matches)
print(f"{rejoined} pairs rejoined, {len(matches) - rejoined} left as they are")
issues["text"] = issues.text.map(rejoin)

A text's **noise rate**: the share of its words (longer than one letter) that `wordfreq` doesn't know. Write a function `noise(text)`, compute it for each issue and compare the two newspapers. Show the corpus's 20 most frequent unknown forms: OCR errors, proper nouns, old spellings?

Many are halves of words split by a space, without a hyphen: « gou vernement », « so cialiste ». The published passages rejoin those too, when one of the two pieces is unknown and the joined word is known.

In [ ]:
# Your code here

### Solution

In [ ]:
def words(text: str) -> list[str]:
    return [word for word in WORD.findall(text) if len(word) > 1]


def noise(text: str) -> float:
    found = words(text)
    return sum(not known(word) for word in found) / max(len(found), 1)


issues["noise"] = issues.text.map(noise)
print(issues.groupby("newspaper").noise.describe())
Counter(
    word.lower() for text in issues.text for word in words(text) if not known(word)
).most_common(20)

A whole issue is too long for a model, and mixes dozens of articles: we split it into **passages**. Write a function `passages(text)` that accumulates an issue's lines until it has at least 100 words, and cuts at the first line that ends a sentence (with `.`, `!`, `?` or `»`). How many passages do you get?

Then compare with `passages.parquet`, the version published for the next labs: it also splits lines longer than 250 words into sentences, and drops the passages whose noise is above 0.15 and the duplicates. How many passages, how long? Read three of them at random.

In [ ]:
# Your code here

### Solution

In [ ]:
END = re.compile(r"[.!?»]\s*$")


def passages(text: str) -> list[str]:
    out, current, n = [], [], 0
    for line in text.split("\n"):
        line = " ".join(line.split())
        if not line:
            continue
        current.append(line)
        n += len(line.split())
        if n >= 100 and END.search(line):
            out.append(" ".join(current))
            current, n = [], 0
    return out


mine = [passage for text in issues.text for passage in passages(text)]
print(len(mine))

published = pd.read_parquet("passages.parquet")
print(published.newspaper.value_counts())
print(published.n_words.describe())
for text in published.text.sample(3, random_state=0):
    print(text, end="\n\n")

## 3. Comparing tokenizers

Load three tokenizers with `transformers.AutoTokenizer.from_pretrained` (the tokenizer alone downloads in seconds):

- `google-bert/bert-base-multilingual-cased` (WordPiece)
- `Qwen/Qwen3-Embedding-0.6B` (byte-level BPE; the embedding model of the next labs)
- `FacebookAI/xlm-roberta-base` (Unigram)

For each, show the tokens (decode each ID with `tokenizer.decode([id])` so they stay readable) and the number of tokens per word of the texts in `samples`. What costs the most tokens?

In [ ]:
samples = {
    "French": "Le conseil municipal a voté hier soir le budget de la nouvelle médiathèque.",
    "English": "The city council voted on the budget for the new media library last night.",
    "proper noun": "Clemenceau et Poincaré",
    "OCR": "Où .av b; aifn'wmant Mme Gueydan, it. o ta î - iu ei est arrivée a la barre",
}

In [ ]:
# Your code here

### Solution

In [ ]:
from transformers import AutoTokenizer

tokenizers = {
    name: AutoTokenizer.from_pretrained(name)
    for name in (
        "google-bert/bert-base-multilingual-cased",
        "Qwen/Qwen3-Embedding-0.6B",
        "FacebookAI/xlm-roberta-base",
    )
}
for name, tokenizer in tokenizers.items():
    print(name)
    for label, text in samples.items():
        ids = tokenizer(text, add_special_tokens=False).input_ids
        # decode() makes each token readable (Qwen3 splits bytes, not characters).
        tokens = [tokenizer.decode([token]) for token in ids]
        print(f"  {label:11} {len(ids) / len(text.split()):.2f} tokens/word {tokens}")

# The noisy OCR text costs the most: its word pieces aren't in the vocabularies. Then
# French, more expensive than English (vocabularies learned on mostly English corpora),
# especially for Qwen3.

How many tokens (for Qwen3's tokenizer) does the published passages corpus hold? Estimate it on a sample of 2,000 passages. At $0.10 per million input tokens (the order of magnitude of a small model through an API), what does one pass of the whole corpus through an LLM cost, input alone?

In [ ]:
# Your code here

### Solution

In [ ]:
tokenizer = tokenizers["Qwen/Qwen3-Embedding-0.6B"]
sample = published.text.sample(2000, random_state=0)
tokens_per_passage = np.mean([len(tokenizer(text).input_ids) for text in sample])
total_tokens = tokens_per_passage * len(published)
print(f"{tokens_per_passage:.0f} tokens per passage, {total_tokens / 1e6:.1f} million in total")
print(f"${total_tokens / 1e6 * 0.10:.2f} for one pass of the corpus")